In [93]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

# Pipelines et préprocessing
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder, PolynomialFeatures
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectKBest, f_classif, RFE
from sklearn.compose import ColumnTransformer

# Métriques et évaluation
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
                             roc_curve, roc_auc_score, precision_recall_curve, auc)

# Gestion des données déséquilibrées
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

# Sauvegarde de modèles
import joblib
import pickle
from scipy.stats import loguniform
import os 

# Charger les données et suppression du doublon
df = pd.read_csv('../data/insurance.csv')
df = df.drop_duplicates().reset_index(drop=True)


print(df.dtypes)
df_encoded = df.copy()

numerical_features = ['age', 'bmi', 'children']  # charges est la cible
categorical_features = ['sex', 'smoker', 'region']




# Pas de transformation ou d'encodage avant le split (évite toute fuite de données "Data Leakage")

X = df_encoded[numerical_features + categorical_features]
y = df_encoded['charges']

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Préprocesseur pas de SimpleImputer strategy=median ou most_frequent pour eviter d'avoir une baisse de qualité de données non visible (et donc fausser les resultats)
preprocessor = ColumnTransformer(
    transformers=[
        ('num', 
         Pipeline([
             ('scaler', StandardScaler())
         ]),
         numerical_features),
        ('cat', 
         Pipeline([
             ('onehot', OneHotEncoder(drop='first', handle_unknown='ignore'))
         ]),
         categorical_features)
    ]
)
# PolynomialFeatures permet de créer des interactions de maniere automatique
def create_pipeline_with_interactions(model, degree=2):
    return Pipeline([
        ('preprocessor', preprocessor),
        ('interactions', PolynomialFeatures(degree=degree, interaction_only=True, include_bias=False)),
        ('regressor', model)
    ])
# Définir les pipelines complets
pipelines = {
    'Linear': create_pipeline_with_interactions(LinearRegression()),
    'Ridge': create_pipeline_with_interactions(Ridge()),
    'Lasso': create_pipeline_with_interactions(Lasso(max_iter=2000))
}

# Grilles d'hyperparamètres !! pour l'instant les meilleurs hyper param sont injecter en dur dans le tableau 0.1879 pr ridge et 62.2003 pour lasso
param_grids = {
    'Linear': {},
    'Ridge': {'regressor__alpha': [0.1879, 10.0, 100.0, 1000.0]},
    'Lasso': {'regressor__alpha': [0.1, 1.0, 10.0, 62.2003]}
}
# Dictionnaire pour stocker les meilleurs paramètres approximatifs

# Grilles larges pour la recherche aléatoire
wide_param_dists = {
    'Ridge': {'regressor__alpha': loguniform(1e-2, 1e4)},
    'Lasso': {'regressor__alpha': loguniform(1e-4, 1e2)}
}
print("wide_param_dists")
coarse_best_params = {}

for name in ['Ridge', 'Lasso']:
    print(f" Randomized search for {name}...")
    pipe = pipelines[name]
    
    random_search = RandomizedSearchCV(
        pipe,
        param_distributions=wide_param_dists[name],
        n_iter=50,
        cv=5,
        scoring='neg_mean_squared_error',
        n_jobs=-1,
        random_state=42
    )
    random_search.fit(X_train, y_train)
    coarse_best_params[name] = random_search.best_params_['regressor__alpha']
    print(f" Meilleur alpha approximatif : {coarse_best_params[name]:.4f}")

# Entraîner avec GridSearchCV (validation croisée intégrée)
best_models = {}
cv_results = {}


for name in pipelines:
    print(f"Optimisation de {name}...")
    grid = GridSearchCV(
        pipelines[name],
        param_grids[name],
        cv=5,
        scoring='neg_mean_squared_error',  # ou 'r2'
        n_jobs=-1
    )
    grid.fit(X_train, y_train)
    
    best_models[name] = grid.best_estimator_
    cv_results[name] = {
        'best_alpha': grid.best_params_.get('regressor__alpha', 'N/A'),
        'best_cv_score (RMSE)': np.sqrt(-grid.best_score_)
    }

    os.makedirs('models', exist_ok=True)
    joblib.dump(grid.best_estimator_, f"models/{name.lower()}_model.joblib")
    print(f"Modèle {name} sauvegardé avec alpha = {grid.best_params_.get('regressoralpha', 'N/A')}")


final_results = {}

for name, model in best_models.items():
    y_pred = model.predict(X_test)
    final_results[name] = {
        'MAE': mean_absolute_error(y_test, y_pred),
        'RMSE': np.sqrt(mean_squared_error(y_test, y_pred)),
        'R²': r2_score(y_test, y_pred),
        'best_alpha': cv_results[name]['best_alpha']
    }

# Affichage
print("\nRésultats finaux sur le jeu de test :")
for name, metrics in final_results.items():
    print(f"\n{name} (α={metrics['best_alpha']}):")
    print(f"  MAE : {metrics['MAE']:.2f}")
    print(f"  RMSE : {metrics['RMSE']:.2f}")
    print(f"  R² : {metrics['R²']:.4f}")

data = []
for name, metrics in final_results.items():
    if name == 'Linear':
        strengths = "Simple, interprétable, pas d'hyperparamètre"
        weaknesses = "Sensible au surapprentissage si features corrélées"
    elif name == 'Ridge':
        strengths = "Stable, gère la multicolinéarité, garde toutes les features"
        weaknesses = "Moins interprétable que Linear, nécessite tuning d'alpha"
    else:
        strengths = "Sélectionne les features utiles, sparse"
        weaknesses = "Peut éliminer des features pertinentes, instable si features corrélées"
    
    data.append({
        'Modèle': name,
        'MAE ($)': round(metrics['MAE'], 2),
        'RMSE ($)': round(metrics['RMSE'], 2),
        'R²': round(metrics['R²'], 4),
        'Alpha': metrics['best_alpha'],
        'Points forts': strengths,
        'Points faibles': weaknesses
    })

df_summary = pd.DataFrame(data)
print(df_summary)

# Créer le tableau
fig = go.Figure(data=[go.Table(
    header=dict(
        values=list(df_summary.columns),
        fill_color='paleturquoise',
        align='left',
        font_size=12,
        height=30
    ),
    cells=dict(
        values=[df_summary[col] for col in df_summary.columns],
        fill_color='lavender',
        align='left',
        font_size=11,
        height=25
    ))
])

fig.update_layout(
    title="Comparatif des modèles de régression",
    title_x=0.5,
    width=1000,
    height=600
)

fig.show()

age           int64
sex          object
bmi         float64
children      int64
smoker       object
region       object
charges     float64
dtype: object
wide_param_dists
 Randomized search for Ridge...
 Meilleur alpha approximatif : 0.1879
 Randomized search for Lasso...
 Meilleur alpha approximatif : 62.2003
Optimisation de Linear...
Modèle Linear sauvegardé avec alpha = N/A
Optimisation de Ridge...
Modèle Ridge sauvegardé avec alpha = N/A
Optimisation de Lasso...
Modèle Lasso sauvegardé avec alpha = N/A

Résultats finaux sur le jeu de test :

Linear (α=N/A):
  MAE : 2822.81
  RMSE : 4636.52
  R² : 0.8830

Ridge (α=0.1879):
  MAE : 2825.72
  RMSE : 4634.68
  R² : 0.8831

Lasso (α=62.2003):
  MAE : 2887.53
  RMSE : 4621.20
  R² : 0.8838
   Modèle  MAE ($)  RMSE ($)      R²    Alpha  \
0  Linear  2822.81   4636.52  0.8830      N/A   
1   Ridge  2825.72   4634.68  0.8831   0.1879   
2   Lasso  2887.53   4621.20  0.8838  62.2003   

                                        Points forts  \

In [78]:
def predict_insurance_charges_v2(
    model_name="ridge",
    age=30,
    bmi=25.0,
    children=0,
    sex="female",
    smoker=False,
    region="southwest"
):
    """
    Prédit les frais médicaux à partir des données BRUTES (comme dans le CSV original).
    """
    import joblib
    import pandas as pd
    import numpy as np

#Charger le modèle,
    pipe = joblib.load(f"models/{model_name.lower()}_model.joblib")

#Créer un DataFrame avec les données BRUTES (comme dans df original),
    client = pd.DataFrame([{
        'age': age,
        'bmi': bmi,
        'children': children,
        'sex': sex,
        'smoker': 'yes' if smoker else 'no',
        'region': region
    }])

#Prédire,
    prediction = pipe.predict(client)[0]

#Optionnel : éviter les négatifs (si tu n'as pas fait log(y)),
    if prediction < 0:
        print(" Attention : prédiction négative ! Envisager log(charges).")

    return prediction

In [79]:
client_args = dict(age=30, bmi=24.0, children=1, sex="female", smoker=False, region="southwest")

for model in ["linear", "ridge", "lasso"]:
    charge = predict_insurance_charges_v2(model_name=model, **client_args)
    print(f"{model.capitalize():>8} : ${charge:,.2f}")

  Linear : $5,372.40
   Ridge : $5,381.62
   Lasso : $6,133.36
